In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score
from preclean import preprocessing
import pickle

In [2]:
df_transac = pd.read_csv('data/train_transaction.csv')
df_id      = pd.read_csv('data/train_identity.csv')
df_train   = preprocessing(df_transac, df_id)

df_test_t  = pd.read_csv('data/test_transaction.csv')
df_test_i  = pd.read_csv('data/test_identity.csv')
df_test    = preprocessing(df_test_t, df_test_i)

 Final shape: (590540, 825)
  Residuals NaN (float) : 275762358
 Final shape: (506691, 463)
  Residuals NaN (float) : 90604215


As it is by time, we have to reset the index of each t

transaction by the time.

In [3]:
df_train_sorted = df_train.sort_values('TransactionDT').reset_index(drop=True)

X_sorted = df_train_sorted.drop(columns=['TransactionID', 'isFraud'])
y_sorted = df_train_sorted['isFraud']

X_test = df_test.drop(columns=['TransactionID'])
X_test = X_test.reindex(columns=X_sorted.columns, fill_value=0)

We use the best parameters found in the Notebook model.ipynb

In [10]:
with open('optuna_study.pkl', 'rb') as f:
    params = pickle.load(f)

print(params)

In [11]:
best_params = params.best_params.copy()


best_params.update({
    'n_estimators':          1000,
    'scale_pos_weight':      len(y_sorted[y_sorted==0]) / len(y_sorted[y_sorted==1]),
    'tree_method':           'hist',
    'eval_metric':           'auc',
    'early_stopping_rounds': 50,
    'random_state':          42,
    'n_jobs':                -1,
})

tscv = TimeSeriesSplit(n_splits=20)
oof  = np.zeros(len(X_sorted))
test_preds = np.zeros(len(X_test))
scores = []

for fold, (idx_train, idx_val) in enumerate(tscv.split(X_sorted)):
    X_tr,  X_val = X_sorted.iloc[idx_train], X_sorted.iloc[idx_val]
    y_tr,  y_val = y_sorted.iloc[idx_train], y_sorted.iloc[idx_val]

    model = XGBClassifier(**best_params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=100)

    oof[idx_val] = model.predict_proba(X_val)[:, 1]
    test_preds  += model.predict_proba(X_test)[:, 1] / tscv.n_splits
    scores.append(roc_auc_score(y_val, oof[idx_val]))
    print(f"Fold {fold+1} AUC : {scores[-1]:.4f}")

print(f"\nAUC moyenne    : {np.mean(scores):.4f}")
print(f"AUC OOF totale : {roc_auc_score(y_sorted, oof):.4f}")

[0]	validation_0-auc:0.74875
[100]	validation_0-auc:0.88767
[200]	validation_0-auc:0.89192
[300]	validation_0-auc:0.89288
[327]	validation_0-auc:0.89276
Fold 1 AUC : 0.8932
[0]	validation_0-auc:0.84837
[100]	validation_0-auc:0.91554
[167]	validation_0-auc:0.91532
Fold 2 AUC : 0.9163
[0]	validation_0-auc:0.82750
[100]	validation_0-auc:0.89979
[200]	validation_0-auc:0.90635
[300]	validation_0-auc:0.91046
[400]	validation_0-auc:0.91126
[453]	validation_0-auc:0.91001
Fold 3 AUC : 0.9114
[0]	validation_0-auc:0.81238
[100]	validation_0-auc:0.89593
[105]	validation_0-auc:0.89662
Fold 4 AUC : 0.8997
[0]	validation_0-auc:0.81054
[100]	validation_0-auc:0.91535
[143]	validation_0-auc:0.91588
Fold 5 AUC : 0.9169
[0]	validation_0-auc:0.78067
[100]	validation_0-auc:0.91905
[121]	validation_0-auc:0.91760
Fold 6 AUC : 0.9210
[0]	validation_0-auc:0.81284
[100]	validation_0-auc:0.92094
[146]	validation_0-auc:0.91842
Fold 7 AUC : 0.9216
[0]	validation_0-auc:0.85592
[100]	validation_0-auc:0.92448
[115]	va

In [13]:
print("═══════════════════════════════════════")
print("Comparison methods")
print("═══════════════════════════════════════")
print(f"StratifiedKFold (notebook model.ipynb): AUC OOF = 0.9715")
print(f"TimeSeriesSplit: AUC OOF = {roc_auc_score(y_sorted, oof):.4f}")
print()
print("The random validation overestimated the real performance on temporal data.")

submission = pd.DataFrame({
    'TransactionID': df_test['TransactionID'],
    'isFraud':       test_preds
})
submission.to_csv('submission_temporal.csv', index=False)

═══════════════════════════════════════
Comparison methods
═══════════════════════════════════════
StratifiedKFold (notebook model.ipynb): AUC OOF = 0.9715
TimeSeriesSplit: AUC OOF = 0.8798

The random validation overestimated the real performance on temporal data.
